# Prompt-based document classification

Compare prompt levels, classification transitions, confidence scores, and the BERT error subsets.

In [ ]:
from shared import paths
from IPython.display import display


## Setup

In [ ]:
paths.prepare_directories()


## Performance

In [ ]:
from prompting.evaluation import evaluate_document

document_scores = evaluate_document(paths.PROMPT_RESULTS / 'document')


## Pairwise prompt comparison

In [ ]:
import os
import pandas as pd
from shared.utils import load_pickle

BASE_DIR = str(paths.PROMPT_RESULTS)
SUBDIR = 'document'
REF_LABEL = 'YES'
TOP_RECIPIENTS = 15
from prompting.analysis_utils import read_file


def choose_files(files):
    print('\nAvailable CSV files:')
    for i, f in enumerate(files):
        print(f'  [{i}] {f}')
    idx1 = int(input('\nSelect FIRST file (baseline): '))
    idx2 = int(input('Select SECOND file (to compare): '))
    return (files[idx1], files[idx2])


def is_ref(x):
    return str(x).strip().upper() == REF_LABEL


def determine_status(row):
    g, l1, l2 = (
        str(row['gold_label']).strip().upper(),
        str(row['label_1']).strip().upper(),
        str(row['label_2']).strip().upper(),
    )
    if l1 == l2:
        return 'unchanged'
    if is_ref(g):
        if is_ref(l1) and (not is_ref(l2)):
            return 'TP → FN'
        elif not is_ref(l1) and is_ref(l2):
            return 'FN → TP'
    elif is_ref(l1) and (not is_ref(l2)):
        return 'FP → TN'
    elif not is_ref(l1) and is_ref(l2):
        return 'TN → FP'
    return 'other_change'


def compare_predictions(file1, file2):
    df1 = read_file(file1)
    df2 = read_file(file2)
    required = {'letter_id', 'label', 'gold_label'}
    if not required.issubset(df1.columns) or not required.issubset(df2.columns):
        raise ValueError(f'Both files must contain columns: {required}')
    merged = pd.merge(
        df1[['letter_id', 'label', 'gold_label']],
        df2[['letter_id', 'label']],
        on='letter_id',
        suffixes=('_1', '_2'),
    )
    merged['change_status'] = merged.apply(determine_status, axis=1)
    diff = merged[merged['change_status'] != 'unchanged'].copy()
    return diff


def enrich_with_metadata(diff_df):
    letters = load_pickle()
    meta = {l.id: {'year': l.year, 'recipient': l.recipient} for l in letters}
    diff_df['year'] = diff_df['letter_id'].map(lambda x: meta.get(x, {}).get('year'))
    diff_df['recipient'] = diff_df['letter_id'].map(lambda x: meta.get(x, {}).get('recipient'))
    return diff_df


def generate_change_mappings(diff_df):
    year_map = diff_df.groupby(['change_status', 'year']).size().reset_index(name='count').sort_values(
        ['change_status', 'year'],
    )
    rec_map = diff_df.groupby(['change_status', 'recipient']).size().reset_index(name='count').sort_values(
        ['change_status', 'count'],
        ascending=[True, False],
    )
    return (year_map, rec_map)


def print_summaries(diff_df, year_map, rec_map):
    print('\n=== OVERALL CHANGE COUNTS ===')
    total_counts = diff_df['change_status'].value_counts().sort_index()
    total = total_counts.sum()
    for k, v in total_counts.items():
        print(f'{k:8s}: {v:3d} ({v / total * 100:5.1f}%)')
    print('\n=== CHANGES BY YEAR ===')
    for ctype, sub in year_map.groupby('change_status'):
        print(f'\n--- {ctype} ---')
        print(sub[['year', 'count']].to_string(index=False))
    print(f'\n=== CHANGES BY RECIPIENT (Top {TOP_RECIPIENTS} per type) ===')
    for ctype, sub in rec_map.groupby('change_status'):
        print(f'\n--- {ctype} ---')
        print(sub.head(TOP_RECIPIENTS)[['recipient', 'count']].to_string(index=False))


def main():
    target_dir = os.path.join(BASE_DIR, SUBDIR)
    files = [f for f in os.listdir(target_dir) if f.endswith('.csv')]
    if not files:
        print(f'No CSV files found in {target_dir}')
        return
    f1_name, f2_name = choose_files(files)
    f1 = os.path.join(target_dir, f1_name)
    f2 = os.path.join(target_dir, f2_name)
    print(f'\nComparing (baseline → comparison):\n  1⃣ {f1_name}\n  2⃣ {f2_name}\n')
    diff_df = compare_predictions(f1, f2)
    if diff_df.empty:
        print(' All predictions identical.')
        return
    print(f' {len(diff_df)} instances changed.\n')
    diff_df = enrich_with_metadata(diff_df)
    year_map, rec_map = generate_change_mappings(diff_df)
    print_summaries(diff_df, year_map, rec_map)
    base_out = f"diff_mapping_{f1_name.replace('.csv', '')}_vs_{f2_name.replace('.csv', '')}"
    detailed_path = os.path.join(target_dir, f'{base_out}_detailed.csv')
    year_path = os.path.join(target_dir, f'{base_out}_by_year.csv')
    rec_path = os.path.join(target_dir, f'{base_out}_by_recipient.csv')
    diff_df.to_csv(detailed_path, index=False)
    year_map.to_csv(year_path, index=False)
    rec_map.to_csv(rec_path, index=False)
    print(f'\n Saved detailed diff with metadata: {detailed_path}')
    print(f' Saved year mapping: {year_path}')
    print(f' Saved recipient mapping: {rec_path}')
if __name__ == '__main__':
    main()


## Transitions across prompt levels

In [ ]:
from __future__ import annotations
import re
from pathlib import Path
from typing import Dict, List, Tuple
import pandas as pd
import plotly.graph_objects as go

BASE_DIR = Path(str(paths.PROMPT_RESULTS / 'document'))
OUT_DIR = Path(str(paths.FIGURES_DIR))
LEVELS = ['L1', 'L2', 'L3', 'L4', 'L5']
CSV_PATTERN = 'llm_results_document_{level}.csv'
OUT_ALL_PNG = OUT_DIR / 'sankey_document_all_TP_FN_TN_FP.png'
OUT_REF_PNG = OUT_DIR / 'sankey_document_ref_only_TP_FN.png'
OUT_NOREF_PNG = OUT_DIR / 'sankey_document_noref_only_TN_FP.png'
ADDITIONS = {
    'L2': '+ metadata',
    'L3': '+ definitions and cues',
    'L4': '+ examples',
    'L5': '+ rationale and evidence',
}
ADDITIONS_Y = -0.02
ADDITIONS_X_SHIFT = -0.19
LEFT_LABEL_X = -0.02
LEFT_LABEL_Y_OVERRIDES = {'TP': 0.82, 'FN': 0.57, 'TN': 0.29, 'FP': 0.04}
LEVEL_HEADER_Y_DEFAULT = 1.06
LEVEL_HEADER_POS_OVERRIDES: Dict[str, Tuple[float | None, float | None]] = {
    'L1': (-0.005, 1.062),
    'L2': (0.24, 1.17),
    'L3': (None, 1.179),
    'L4': (0.76, 1.17),
    'L5': (1.01, 1.172),
}
FONT_BASE = 28
FONT_TITLE = 28
FONT_LEVEL = 28
FONT_LEFT = 28
FONT_ADDITIONS = 28
FONT_SANKEY_LABELS = 26


def _normalise_yesno(x) -> str | None:
    if x is None:
        return None
    s = str(x).strip().upper()
    return s if s in {'YES', 'NO'} else None


def _fallback_parse_yesno(raw: str) -> str | None:
    if not raw:
        return None
    m = re.search('\\b(YES|NO)\\b', str(raw).upper())
    return m.group(1) if m else None


def load_level(level: str) -> pd.DataFrame:
    path = BASE_DIR / CSV_PATTERN.format(level=level)
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')
    df = pd.read_csv(path)
    required = {'letter_id', 'gold_label'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{path.name} is missing required columns: {sorted(missing)}')
    if 'label' in df.columns:
        pred = df['label'].apply(_normalise_yesno)
    else:
        pred = pd.Series([None] * len(df), index=df.index)
    if pred.isna().all() and 'raw_response' in df.columns:
        pred = df['raw_response'].apply(_fallback_parse_yesno).apply(_normalise_yesno)
    gold = df['gold_label'].apply(_normalise_yesno)
    out = pd.DataFrame({'letter_id': df['letter_id'].astype(str), 'gold': gold, 'pred': pred}).dropna(
        subset=['gold', 'pred'],
    ).copy()
    out['level'] = level
    return out


def outcome(gold: str, pred: str) -> str:
    if gold == 'YES' and pred == 'YES':
        return 'TP'
    if gold == 'YES' and pred == 'NO':
        return 'FN'
    if gold == 'NO' and pred == 'YES':
        return 'FP'
    if gold == 'NO' and pred == 'NO':
        return 'TN'
    raise ValueError(f'Unexpected gold/pred pair: {gold}/{pred}')


def build_outcome_table(level_dfs: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    wide = None
    for level in LEVELS:
        df = level_dfs[level]
        tmp = df.copy()
        tmp['outcome'] = [outcome(g, p) for g, p in zip(tmp['gold'], tmp['pred'])]
        tmp = tmp[['letter_id', 'gold', 'outcome']].drop_duplicates(subset=['letter_id'])
        tmp = tmp.rename(columns={'outcome': level})
        if wide is None:
            wide = tmp
        else:
            wide = wide.merge(tmp[['letter_id', level]], on='letter_id', how='inner')
    if wide is None or wide.empty:
        raise RuntimeError('No data loaded or intersection across levels is empty.')
    return wide
STATE_COLOURS_NODE = {
    'TP': 'rgba(44, 160, 44, 0.85)',
    'FN': 'rgba(255, 127, 14, 0.85)',
    'TN': 'rgba(31, 119, 180, 0.85)',
    'FP': 'rgba(214, 39, 40, 0.85)',
}
STATE_COLOURS_LINK = {
    'TP': 'rgba(44, 160, 44, 0.35)',
    'FN': 'rgba(255, 127, 14, 0.35)',
    'TN': 'rgba(31, 119, 180, 0.35)',
    'FP': 'rgba(214, 39, 40, 0.35)',
}


def _node_positions_fixed(levels: List[str], states: List[str]) -> Tuple[List[float], List[float]]:
    x, y = ([], [])
    if len(states) == 1:
        y_vals = [0.5]
    else:
        lo, hi = (0.06, 0.94)
        step = (hi - lo) / (len(states) - 1)
        y_vals = [lo + i * step for i in range(len(states))]
    for i, _lvl in enumerate(levels):
        for j, _st in enumerate(states):
            x.append(i / max(1, len(levels) - 1))
            y.append(y_vals[j])
    return (x, y)


def _layout_annotations(levels: List[str], states: List[str]):
    ann = []
    for i, lvl in enumerate(levels):
        x_default = i / max(1, len(levels) - 1)
        y_default = LEVEL_HEADER_Y_DEFAULT
        x_override, y_override = LEVEL_HEADER_POS_OVERRIDES.get(lvl, (None, None))
        x = x_override if x_override is not None else x_default
        y = y_override if y_override is not None else y_default
        ann.append(
            dict(
                x=x,
                y=y,
                xref='paper',
                yref='paper',
                text=f'<b>{lvl}</b>',
                showarrow=False,
                font=dict(size=FONT_LEVEL),
            ),
        )
    _, y_positions = _node_positions_fixed(levels, states)
    row_y_nodes = y_positions[:len(states)]
    for st, yy_node in zip(states, row_y_nodes):
        yy_paper_default = 1.0 - yy_node
        yy_paper = LEFT_LABEL_Y_OVERRIDES.get(st, yy_paper_default)
        ann.append(
            dict(
                x=LEFT_LABEL_X,
                y=yy_paper,
                xref='paper',
                yref='paper',
                text=f'<b>{st}</b>',
                showarrow=False,
                xanchor='right',
                yanchor='middle',
                font=dict(size=FONT_LEFT),
            ),
        )
    for i in range(1, len(levels)):
        lvl = levels[i]
        if lvl not in ADDITIONS:
            continue
        x_col = i / max(1, len(levels) - 1)
        ann.append(
            dict(
                x=x_col + ADDITIONS_X_SHIFT,
                y=ADDITIONS_Y,
                xref='paper',
                yref='paper',
                text=ADDITIONS[lvl],
                showarrow=False,
                font=dict(size=FONT_ADDITIONS),
                xanchor='left',
                yanchor='top',
            ),
        )
    return ann


def _format_count_label(level_idx: int, current: int, prev: int | None) -> str:
    if level_idx == 0 or prev is None:
        return f'{current} instances'
    delta = current - prev
    sign = '+' if delta > 0 else ''
    return f'{current} instances<br>({sign}{delta})'


def _build_links_and_nodes(wide: pd.DataFrame, levels: List[str], states: List[str], subset_mask):
    data = wide.loc[subset_mask].copy()
    node_keys = [(lvl, st) for lvl in levels for st in states]
    node_index = {k: i for i, k in enumerate(node_keys)}
    node_counts: Dict[Tuple[str, str], int] = {(lvl, st): int((data[lvl] == st).sum()) for lvl, st in node_keys}
    node_labels: List[str] = []
    for li, lvl in enumerate(levels):
        prev_lvl = levels[li - 1] if li > 0 else None
        for st in states:
            cur = node_counts[lvl, st]
            prev = node_counts[prev_lvl, st] if prev_lvl is not None else None
            node_labels.append(_format_count_label(li, cur, prev))
    node_colors = [STATE_COLOURS_NODE.get(st, 'rgba(150,150,150,0.85)') for _lvl, st in node_keys]
    sources, targets, values, link_colors = ([], [], [], [])
    for i in range(len(levels) - 1):
        a, b = (levels[i], levels[i + 1])
        idx = pd.MultiIndex.from_product([states, states], names=[a, b])
        counts = data.groupby([a, b]).size().reindex(idx, fill_value=0).reset_index(name='n')
        for _, row in counts.iterrows():
            sa = row[a]
            sb = row[b]
            n = int(row['n'])
            if n == 0:
                continue
            sources.append(node_index[a, sa])
            targets.append(node_index[b, sb])
            values.append(n)
            link_colors.append(STATE_COLOURS_LINK.get(sa, 'rgba(150,150,150,0.25)'))
    x, y = _node_positions_fixed(levels, states)
    return (node_labels, node_colors, sources, targets, values, link_colors, x, y)


def save_sankey_static_png(
    out_png: Path,
    wide: pd.DataFrame,
    *,
    title: str,
    states: List[str],
    subset_mask,
    width: int,
    height: int,
    scale: int=2,
):
    node_labels, node_colors, sources, targets, values, link_colors, x, y = _build_links_and_nodes(wide=wide, levels=LEVELS, states=states, subset_mask=subset_mask)
    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement='fixed',
                textfont=dict(size=FONT_SANKEY_LABELS),
                node=dict(label=node_labels, color=node_colors, pad=18, thickness=18, x=x, y=y),
                link=dict(source=sources, target=targets, value=values, color=link_colors),
            ),
        ],
    )
    fig.update_layout(
        width=width,
        height=height,
        font=dict(size=FONT_BASE),
        margin=dict(l=160, r=40, t=150, b=240),
        annotations=_layout_annotations(LEVELS, states),
    )
    fig.write_image(str(out_png), width=width, height=height, scale=scale)
    print(f'Wrote: {out_png}')


def print_level_counts(wide: pd.DataFrame, levels: List[str]) -> None:
    print('\n=== Per-level outcome counts (intersection set) ===')
    for lvl in levels:
        vc = wide[lvl].value_counts()
        tp = int(vc.get('TP', 0))
        fn = int(vc.get('FN', 0))
        tn = int(vc.get('TN', 0))
        fp = int(vc.get('FP', 0))
        total = tp + fn + tn + fp
        print(f'{lvl}: TP={tp}  FN={fn}  TN={tn}  FP={fp}  (total={total})')


def print_subset_counts(
    wide: pd.DataFrame,
    levels: List[str],
    mask,
    states: List[str],
    header: str,
) -> None:
    print(f'\n=== {header} ===')
    sub = wide.loc[mask].copy()
    for lvl in levels:
        vc = sub[lvl].value_counts().reindex(states, fill_value=0).astype(int)
        pairs = '  '.join([f'{st}={int(vc.loc[st])}' for st in states])
        print(f'{lvl}: {pairs}  (total={int(vc.sum())})')


def main():
    level_dfs: Dict[str, pd.DataFrame] = {lvl: load_level(lvl) for lvl in LEVELS}
    wide = build_outcome_table(level_dfs)
    print_level_counts(wide, LEVELS)
    print('\n=== Gold label distribution (intersection set) ===')
    print(wide['gold'].value_counts().to_dict())
    all_states = ['TP', 'FN', 'TN', 'FP']
    save_sankey_static_png(
        out_png=OUT_ALL_PNG,
        wide=wide,
        title='Document level: transitions between prompt levels (TP/FN/TN/FP)',
        states=all_states,
        subset_mask=pd.Series([True] * len(wide), index=wide.index),
        width=2200,
        height=900,
        scale=2,
    )
    mask_ref = wide['gold'].eq('YES')
    print_subset_counts(
        wide,
        LEVELS,
        mask_ref,
        states=['TP', 'FN'],
        header='gold=YES subset counts (TP/FN)',
    )
    save_sankey_static_png(
        out_png=OUT_REF_PNG,
        wide=wide,
        title='Document level (gold=Ref): transitions between prompt levels (TP/FN)',
        states=['TP', 'FN'],
        subset_mask=mask_ref,
        width=2200,
        height=720,
        scale=2,
    )
    mask_noref = wide['gold'].eq('NO')
    print_subset_counts(
        wide,
        LEVELS,
        mask_noref,
        states=['TN', 'FP'],
        header='gold=NO subset counts (TN/FP)',
    )
    save_sankey_static_png(
        out_png=OUT_NOREF_PNG,
        wide=wide,
        title='Document level (gold=NoRef): transitions between prompt levels (TN/FP)',
        states=['TN', 'FP'],
        subset_mask=mask_noref,
        width=2200,
        height=720,
        scale=2,
    )
    print('\nInstances used (intersection across all prompt levels):', len(wide))
    print('Saved figures to:', OUT_DIR)
if __name__ == '__main__':
    main()


## Label-token confidence

In [ ]:
from __future__ import annotations
import os
import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import pandas as pd

BASE_DIR = Path(str(paths.PROMPT_RESULTS))
SUBDIR = 'document'
POS_LABEL = 'YES'
LEVELS = ['L1', 'L2', 'L3', 'L4', 'L5']
WRITE_SUMMARIES = True
OUT_DIR = Path(str(paths.FIGURES_DIR / 'llm_confidence_summaries_document'))
OUT_DIR.mkdir(parents=True, exist_ok=True)
from prompting.analysis_utils import read_file


def extract_confidence(slot_json_str: str) -> Optional[float]:
    try:
        data = json.loads(slot_json_str)
        if not data:
            return None
        slot = data[0]
        return float(slot.get('prob', 0.0))
    except Exception:
        return None


def determine_outcome(row: pd.Series) -> str:
    pred = str(row['label']).strip().upper()
    gold = str(row['gold_label']).strip().upper()
    if pred == POS_LABEL and gold == POS_LABEL:
        return 'TP'
    if pred == POS_LABEL and gold != POS_LABEL:
        return 'FP'
    if pred != POS_LABEL and gold != POS_LABEL:
        return 'TN'
    if pred != POS_LABEL and gold == POS_LABEL:
        return 'FN'
    return 'OTHER'


def compute_outcome_stats(df: pd.DataFrame) -> pd.DataFrame:
    stats = df.groupby('outcome')['confidence'].agg(
        count='count',
        mean='mean',
        std='std',
        min='min',
        max='max',
    ).round(
        3,
    ).sort_index()
    return stats


def compute_global_stats(df: pd.DataFrame) -> Tuple[float, float, int]:
    return (float(df['confidence'].mean()), float(df['confidence'].std()), int(len(df)))


def load_level_df(path: Path, level: str) -> pd.DataFrame:
    df = read_file(path)
    required = {'slot_probs_json', 'label', 'gold_label', 'letter_id'}
    if not required.issubset(df.columns):
        missing = sorted(required - set(df.columns))
        raise ValueError(f'{path.name} is missing required columns: {missing}')
    df['confidence'] = df['slot_probs_json'].apply(extract_confidence)
    df['outcome'] = df.apply(determine_outcome, axis=1)
    df['level'] = level
    df = df.dropna(subset=['confidence']).copy()
    df['confidence'] = df['confidence'].astype(float)
    return df


def find_level_files(target_dir: Path) -> Dict[str, Path]:
    mapping: Dict[str, Path] = {}
    for lvl in LEVELS:
        f = target_dir / f'llm_results_document_{lvl}.csv'
        if f.exists():
            mapping[lvl] = f
    return mapping


def print_section(title: str) -> None:
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))


def print_lowest(df: pd.DataFrame, n: int=20) -> None:
    cols = ['level', 'letter_id', 'label', 'gold_label', 'confidence', 'outcome']
    cols = [c for c in cols if c in df.columns]
    print(df.nsmallest(n, 'confidence')[cols].to_string(index=False))


def main() -> None:
    target_dir = BASE_DIR / SUBDIR
    if not target_dir.exists():
        print(f' Target directory not found: {target_dir}')
        return
    level_files = find_level_files(target_dir)
    if not level_files:
        print(f' No level files found in {target_dir}')
        print('Expected: llm_results_document_L1.csv ... llm_results_document_L5.csv')
        return
    print_section('FILES FOUND')
    for lvl in LEVELS:
        if lvl in level_files:
            print(f'  {lvl}: {level_files[lvl].name}')
        else:
            print(f'  {lvl}: (missing)')
    per_level_dfs: Dict[str, pd.DataFrame] = {}
    for lvl, path in level_files.items():
        try:
            per_level_dfs[lvl] = load_level_df(path, lvl)
        except Exception as e:
            print(f' Failed to load {path.name}: {e}')
            return
    pooled = pd.concat([per_level_dfs[lvl] for lvl in sorted(per_level_dfs.keys())], ignore_index=True)
    print_section('PER-LEVEL CONFIDENCE SUMMARY')
    per_level_rows: List[Dict[str, object]] = []
    for lvl in [l for l in LEVELS if l in per_level_dfs]:
        df = per_level_dfs[lvl]
        mean_c, std_c, n = compute_global_stats(df)
        per_level_rows.append(
            {'level': lvl, 'n': n, 'mean_conf': round(mean_c, 3), 'std_conf': round(std_c, 3)},
        )
        print(f'\n[{lvl}] Overall confidence: {mean_c:.3f} ± {std_c:.3f} (n={n})')
        stats = compute_outcome_stats(df)
        print('Confidence by outcome:')
        print(stats.to_string())
        print('\nLowest-confidence predictions:')
        print_lowest(df, n=20)
    per_level_summary = pd.DataFrame(per_level_rows)
    print_section('OVERALL (POOLED ACROSS LEVELS) CONFIDENCE SUMMARY')
    mean_c, std_c, n = compute_global_stats(pooled)
    print(f'Overall confidence: {mean_c:.3f} ± {std_c:.3f} (n={n})')
    pooled_stats = compute_outcome_stats(pooled)
    print('\nConfidence by outcome (pooled):')
    print(pooled_stats.to_string())
    print('\nLowest-confidence predictions (pooled):')
    print_lowest(pooled, n=20)
    if WRITE_SUMMARIES:
        out1 = OUT_DIR / 'confidence_summary_per_level.csv'
        per_level_summary.to_csv(out1, index=False)
        rows = []
        for lvl in [l for l in LEVELS if l in per_level_dfs]:
            stats = compute_outcome_stats(per_level_dfs[lvl]).reset_index()
            stats.insert(0, 'level', lvl)
            rows.append(stats)
        per_level_outcome_summary = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
        out2 = OUT_DIR / 'confidence_summary_per_level_by_outcome.csv'
        per_level_outcome_summary.to_csv(out2, index=False)
        out3 = OUT_DIR / 'confidence_summary_pooled_by_outcome.csv'
        pooled_stats.reset_index().to_csv(out3, index=False)
        print_section('WROTE SUMMARY FILES')
        print(f'- {out1}')
        print(f'- {out2}')
        print(f'- {out3}')
        print(f'Output directory: {OUT_DIR}')
    print_section('ID COVERAGE CHECK (OPTIONAL)')
    id_sets = {lvl: set(df['letter_id'].astype(str)) for lvl, df in per_level_dfs.items()}
    common = set.intersection(*id_sets.values()) if len(id_sets) > 1 else next(iter(id_sets.values()))
    union = set.union(*id_sets.values()) if len(id_sets) > 1 else next(iter(id_sets.values()))
    print(f'Common letter_ids across available levels: {len(common)}')
    print(f'Union of letter_ids across available levels:  {len(union)}')
    if len(common) != len(union):
        for lvl in [l for l in LEVELS if l in id_sets]:
            missing = union - id_sets[lvl]
            if missing:
                print(f'  {lvl}: missing {len(missing)} ids (example: {next(iter(missing))})')
if __name__ == '__main__':
    main()


## Repeated label changes

In [ ]:
from __future__ import annotations
import json
from pathlib import Path
from typing import Optional, Dict, List, Tuple
import pandas as pd

BASE_DIR = Path(str(paths.PROMPT_RESULTS / 'document'))
LEVELS = ['L1', 'L2', 'L3', 'L4', 'L5']
POS_LABEL = 'YES'
MIN_SWITCHES = 3
PRINT_TOP_N = 200


def extract_confidence(slot_json_str: str) -> Optional[float]:
    try:
        data = json.loads(slot_json_str)
        if not data:
            return None
        return float(data[0].get('prob', 0.0))
    except Exception:
        return None


def load_level(level: str) -> pd.DataFrame:
    p = BASE_DIR / f'llm_results_document_{level}.csv'
    if not p.exists():
        raise FileNotFoundError(f'Missing file: {p}')
    df = pd.read_csv(p)
    df.columns = [c.strip() for c in df.columns]
    required = {'letter_id', 'label', 'gold_label', 'slot_probs_json'}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f'{p.name} missing required columns: {missing}')
    df['level'] = level
    df['confidence'] = df['slot_probs_json'].apply(extract_confidence)
    df['label'] = df['label'].astype(str).str.strip().str.upper()
    df['gold_label'] = df['gold_label'].astype(str).str.strip().str.upper()
    return df[['letter_id', 'level', 'label', 'gold_label', 'confidence']]


def count_switches(seq: List[str]) -> int:
    switches = 0
    for a, b in zip(seq, seq[1:]):
        if a != b:
            switches += 1
    return switches


def main() -> None:
    dfs = [load_level(lvl) for lvl in LEVELS]
    long_df = pd.concat(dfs, ignore_index=True)
    label_wide = long_df.pivot(index='letter_id', columns='level', values='label')
    gold_wide = long_df.pivot(index='letter_id', columns='level', values='gold_label')
    conf_wide = long_df.pivot(index='letter_id', columns='level', values='confidence')
    complete_mask = label_wide[LEVELS].notna().all(axis=1)
    label_wide = label_wide.loc[complete_mask, LEVELS]
    gold_wide = gold_wide.loc[complete_mask, LEVELS]
    conf_wide = conf_wide.loc[complete_mask, LEVELS]
    patterns = label_wide.apply(lambda r: [r[lvl] for lvl in LEVELS], axis=1)
    switches = patterns.apply(count_switches)
    distinct = patterns.apply(lambda xs: len(set(xs)))
    conf_min = conf_wide.min(axis=1)
    conf_mean = conf_wide.mean(axis=1)
    res = pd.DataFrame(
        {
            'letter_id': label_wide.index,
            'switches': switches.values,
            'distinct_labels': distinct.values,
            'pattern': patterns.apply(lambda xs: ' -> '.join(xs)).values,
            'min_conf': conf_min.values,
            'mean_conf': conf_mean.values,
            'gold_label': gold_wide[LEVELS[0]].values,
        },
    )
    res = res[res['switches'] >= MIN_SWITCHES].copy()
    res = res.sort_values(['switches', 'min_conf'], ascending=[False, True])
    print('\n==============================')
    print('LETTERS WITH FREQUENT PREDICTION FLIPS (DOCUMENT TASK)')
    print('==============================')
    print(f"Levels: {', '.join(LEVELS)}")
    print(f'Criterion: switches >= {MIN_SWITCHES} (counted between consecutive levels)')
    print(f'Letters analysed (complete across all levels): {len(complete_mask[complete_mask])}')
    print(f'Letters matching criterion: {len(res)}\n')
    if res.empty:
        print('(None)')
        return
    out = res.copy()
    out['min_conf'] = out['min_conf'].map(lambda x: '' if pd.isna(x) else f'{x:.6f}')
    out['mean_conf'] = out['mean_conf'].map(lambda x: '' if pd.isna(x) else f'{x:.6f}')
    out = out.head(PRINT_TOP_N)
    out = out[['letter_id', 'gold_label', 'switches', 'distinct_labels', 'pattern', 'min_conf', 'mean_conf']]
    print(out.to_string(index=False))
    print('\n------------------------------')
    print('Switch-count distribution (among matching letters)')
    print('------------------------------')
    dist = res['switches'].value_counts().sort_index()
    for k, v in dist.items():
        print(f'  switches={k}: {v}')
    print("\nNote: 'switches' counts changes between consecutive levels (L1->L2, L2->L3, ...).")
    print('Max possible with 5 levels is 4 switches.')
if __name__ == '__main__':
    main()


## BERT error subsets

In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Dict, Set, Tuple
import pandas as pd

BERT_CSV = Path(str(paths.document_run() / 'general/all_folds_predictions.csv'))
LLM_DIR = Path(str(paths.PROMPT_RESULTS / 'document'))
LEVELS = ['L1', 'L2', 'L3', 'L4', 'L5']
LLM_PATTERN = 'llm_results_document_{level}.csv'
OUT_DIR = LLM_DIR / 'bert_error_subset_eval'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FP_IDS = OUT_DIR / 'bert_fp_letter_ids.txt'
OUT_FN_IDS = OUT_DIR / 'bert_fn_letter_ids.txt'
OUT_SUMMARY = OUT_DIR / 'eval_summary_by_level.csv'
OUT_DETAILS = OUT_DIR / 'eval_details_by_level.csv'
FILTER_VALID_ONLY = True


def normalise_yes_no(x) -> int:
    if pd.isna(x):
        raise ValueError('Encountered NA label where YES/NO was expected.')
    s = str(x).strip().upper()
    if s == 'YES':
        return 1
    if s == 'NO':
        return 0
    raise ValueError(f'Unmappable YES/NO label: {x!r}')


def normalise_01(x) -> int:
    if pd.isna(x):
        raise ValueError('Encountered NA label where 0/1 was expected.')
    try:
        v = int(float(x))
    except Exception as e:
        raise ValueError(f'Unmappable 0/1 label: {x!r}') from e
    if v not in (0, 1):
        raise ValueError(f'Out-of-range 0/1 label: {x!r}')
    return v


def confusion(y_true: pd.Series, y_pred: pd.Series) -> Dict[str, int]:
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    return {'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn}


def safe_div(a: float, b: float) -> float:
    return float(a / b) if b else 0.0


def metrics(y_true: pd.Series, y_pred: pd.Series) -> Dict[str, float]:
    c = confusion(y_true, y_pred)
    tp, tn, fp, fn = (c['TP'], c['TN'], c['FP'], c['FN'])
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * precision * recall, precision + recall) if precision + recall else 0.0
    accuracy = safe_div(tp + tn, tp + tn + fp + fn)
    return {'precision': precision, 'recall': recall, 'f1': f1, 'accuracy': accuracy}


def load_bert_error_sets(path: Path) -> Tuple[Set[str], Set[str], Set[str]]:
    df = pd.read_csv(path)
    required = {'letter_id', 'true_label', 'model_predictions_full_dataset'}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(
            f'BERT CSV missing required columns: {sorted(missing)}\nAvailable: {list(df.columns)}',
        )
    tmp = df[['letter_id', 'true_label', 'model_predictions_full_dataset']].copy()
    tmp['letter_id'] = tmp['letter_id'].astype(str)
    tmp['y_true'] = tmp['true_label'].map(normalise_01)
    tmp['y_pred'] = tmp['model_predictions_full_dataset'].map(normalise_01)
    fp_ids = set(tmp.loc[(tmp['y_true'] == 0) & (tmp['y_pred'] == 1), 'letter_id'].tolist())
    fn_ids = set(tmp.loc[(tmp['y_true'] == 1) & (tmp['y_pred'] == 0), 'letter_id'].tolist())
    union_ids = fp_ids | fn_ids
    return (fp_ids, fn_ids, union_ids)


def load_llm_level(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {'letter_id', 'gold_label', 'label'}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(
            f'{path.name} missing required columns: {sorted(missing)}\nAvailable: {list(df.columns)}',
        )
    if FILTER_VALID_ONLY and 'valid' in df.columns:
        df = df[df['valid'] == True].copy()
    out = df[['letter_id', 'gold_label', 'label']].copy()
    out['letter_id'] = out['letter_id'].astype(str)
    out['y_true'] = out['gold_label'].map(normalise_yes_no)
    out['y_pred'] = out['label'].map(normalise_yes_no)
    return out[['letter_id', 'y_true', 'y_pred']]


def eval_subset(llm_df: pd.DataFrame, subset_ids: Set[str]) -> Tuple[Dict[str, float], Dict[str, int]]:
    sub = llm_df[llm_df['letter_id'].isin(subset_ids)].copy()
    if sub.empty:
        return (
            {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.0},
            {'TP': 0, 'TN': 0, 'FP': 0, 'FN': 0, 'n': 0},
        )
    y_true = sub['y_true']
    y_pred = sub['y_pred']
    m = metrics(y_true, y_pred)
    c = confusion(y_true, y_pred)
    c['n'] = int(len(sub))
    return (m, c)


def write_id_list(path: Path, ids: Set[str]) -> None:
    with path.open('w', encoding='utf-8') as f:
        for _id in sorted(ids):
            f.write(f'{_id}\n')


def print_console_report(summary_df: pd.DataFrame, details_df: pd.DataFrame) -> None:
    pd.set_option('display.width', 200)
    pd.set_option('display.max_columns', 50)
    merged = summary_df.merge(details_df, on=['level', 'subset', 'n'], how='inner', validate='one_to_one')
    for col in ['precision', 'recall', 'f1', 'accuracy']:
        merged[col] = merged[col].map(lambda x: f'{x:.3f}')
    subset_order = ['bert_FP_only', 'bert_FN_only', 'bert_FP_or_FN']
    level_order = LEVELS
    for subset in subset_order:
        print('\n' + '=' * 80)
        print(f'Subset: {subset}')
        print('=' * 80)
        block = merged[merged['subset'] == subset].copy()
        block['level'] = pd.Categorical(block['level'], categories=level_order, ordered=True)
        block = block.sort_values('level')
        cols = ['level', 'n', 'precision', 'recall', 'f1', 'accuracy', 'TP', 'TN', 'FP', 'FN']
        print(block[cols].to_string(index=False))


def main() -> None:
    fp_ids, fn_ids, union_ids = load_bert_error_sets(BERT_CSV)
    print(f'BERT FP: {len(fp_ids)}')
    print(f'BERT FN: {len(fn_ids)}')
    print(f'BERT FP∪FN: {len(union_ids)}')
    write_id_list(OUT_FP_IDS, fp_ids)
    write_id_list(OUT_FN_IDS, fn_ids)
    summary_rows = []
    detail_rows = []
    subsets = [('bert_FP_only', fp_ids), ('bert_FN_only', fn_ids), ('bert_FP_or_FN', union_ids)]
    for lvl in LEVELS:
        llm_path = LLM_DIR / LLM_PATTERN.format(level=lvl)
        if not llm_path.exists():
            raise FileNotFoundError(f'Missing LLM file: {llm_path}')
        llm_df = load_llm_level(llm_path)
        for subset_name, subset_ids in subsets:
            m, c = eval_subset(llm_df, subset_ids)
            summary_rows.append({'level': lvl, 'subset': subset_name, 'n': c['n'], **m})
            detail_rows.append(
                {
                    'level': lvl,
                    'subset': subset_name,
                    'n': c['n'],
                    'TP': c['TP'],
                    'TN': c['TN'],
                    'FP': c['FP'],
                    'FN': c['FN'],
                },
            )
    summary_df = pd.DataFrame(summary_rows)
    details_df = pd.DataFrame(detail_rows)
    subset_order = ['bert_FP_only', 'bert_FN_only', 'bert_FP_or_FN']
    summary_df['level'] = pd.Categorical(summary_df['level'], categories=LEVELS, ordered=True)
    details_df['level'] = pd.Categorical(details_df['level'], categories=LEVELS, ordered=True)
    summary_df['subset'] = pd.Categorical(summary_df['subset'], categories=subset_order, ordered=True)
    details_df['subset'] = pd.Categorical(details_df['subset'], categories=subset_order, ordered=True)
    summary_df = summary_df.sort_values(['subset', 'level']).reset_index(drop=True)
    details_df = details_df.sort_values(['subset', 'level']).reset_index(drop=True)
    summary_df.to_csv(OUT_SUMMARY, index=False)
    details_df.to_csv(OUT_DETAILS, index=False)
    print('\nWrote:')
    print(f'  {OUT_FP_IDS}')
    print(f'  {OUT_FN_IDS}')
    print(f'  {OUT_SUMMARY}')
    print(f'  {OUT_DETAILS}')
    if FILTER_VALID_ONLY:
        print(
            '\nNote: FILTER_VALID_ONLY=True (only rows with valid==True were evaluated where available).',
        )
    print_console_report(summary_df, details_df)
if __name__ == '__main__':
    main()
